# Single Agent Pipeline Project

## Problem Statement
Build a **Single-Agent Smart Assistant** that:
- Understands user queries
- Routes tasks based on intent
- Uses tools when required
- Returns structured JSON output

### The agent should handle:
- Math queries → Calculator Tool
- Keyword extraction → Keyword Tool
- General queries → Direct response

### What You Need to Implement
- Agent logic
- Conditional routing
- Tool integration
- Basic error handling

### Bonus
- Improve routing
- Add logging
- Add more tools


In [7]:
# TOOL 1: Calculator

def calculator(expression: str) -> str:
    """Evaluate a mathematical expression safely (restricted eval, no builtins)."""
    try:
        allowed_chars = set("0123456789+-*/(). %")
        if not expression or not set(expression) <= allowed_chars:
            return "Error in calculation"
        return str(eval(expression, {"__builtins__": {}}, {}))
    except Exception:
        return "Error in calculation"

In [8]:
# TOOL 2: Keyword Extractor

def extract_keywords(text: str) -> list:
    """Extract keywords from text."""
    try:
        words = text.split()
        keywords = list(set([w.lower() for w in words if len(w) > 4]))
        return keywords[:5]
    except Exception:
        return []

## Implement Agent Logic Below

 Use conditional routing:
- If query contains "calculate" → use calculator
- If query contains "keywords" → use keyword extractor
- Else → general response

In [9]:
# AGENT FUNCTION

import re
import logging

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
logger = logging.getLogger("agent")

MATH_TRIGGERS = ["calculate", "compute", "solve", "evaluate", "what is", "what's",
                  "sum of", "add", "plus", "minus", "multiply", "multiplied",
                  "divide", "divided", "product of", "result of"]

KEYWORD_TRIGGERS = ["keyword", "keywords", "key term", "key terms",
                     "extract terms", "main terms", "important words"]

MATH_EXPRESSION_PATTERN = re.compile(
    r"-?\d+(\.\d+)?\s*[-+*/%]\s*-?\d+(\.\d+)?(\s*[-+*/%]\s*-?\d+(\.\d+)?)*"
)


def safe_eval(expr: str):
    """Evaluate a restricted arithmetic expression safely (no builtins, no names)."""
    allowed_chars = set("0123456789+-*/(). %")
    if not expr or not set(expr) <= allowed_chars:
        raise ValueError("Expression contains disallowed characters")
    return eval(expr, {"__builtins__": {}}, {})


def looks_like_math(query: str, query_lower: str) -> bool:
    """True if the query contains an arithmetic expression, OR names a math
    operation while also containing at least one digit (so generic phrases like
    'what is' don't false-positive on non-math questions like 'what is machine learning?')."""
    if MATH_EXPRESSION_PATTERN.search(query):
        return True
    has_digit = bool(re.search(r"\d", query))
    has_math_trigger = any(trigger in query_lower for trigger in MATH_TRIGGERS)
    return has_digit and has_math_trigger


def looks_like_keyword_request(query_lower: str) -> bool:
    return any(trigger in query_lower for trigger in KEYWORD_TRIGGERS)


def extract_expression(query: str) -> str:
    """Pull the numeric/arithmetic portion out of a natural-language query."""
    match = MATH_EXPRESSION_PATTERN.search(query)
    if match:
        return match.group(0).strip()
    match = re.search(r"-?\d+(\.\d+)?", query)
    if not match:
        raise ValueError("No numeric expression found in query")
    return match.group(0).strip()


def clean_text_for_keywords(query: str) -> str:
    """Strip trigger phrases so they don't pollute the extracted keywords."""
    cleaned = query
    lower = cleaned.lower()
    all_triggers = KEYWORD_TRIGGERS + ["extract", "from", "give me", "find the", "of this text"]
    for phrase in sorted(all_triggers, key=len, reverse=True):
        idx = lower.find(phrase)
        if idx != -1:
            cleaned = cleaned[:idx] + cleaned[idx + len(phrase):]
            lower = cleaned.lower()
    return cleaned.strip()


# TOOL 3: General Response

GENERAL_KNOWLEDGE_BASE = {
    "machine learning": "Machine learning is a field of AI where systems learn patterns from data to make predictions or decisions, rather than following explicit rules.",
    "artificial intelligence": "Artificial intelligence is the broader field of building systems that can perform tasks normally requiring human intelligence, like reasoning, perception, and language.",
    "deep learning": "Deep learning is a subset of machine learning that uses multi-layered neural networks to learn complex patterns directly from raw data.",
    "neural network": "A neural network is a model made of layers of connected nodes (neurons) that learn to map inputs to outputs by adjusting connection weights during training.",
    "agent": "An agent is a system that perceives its environment (via input/queries), decides on an action (often by routing to a tool), and executes that action to produce a result.",
    "python": "Python is a high-level, general-purpose programming language known for its readability and wide use in AI, web development, and automation.",
    "api": "An API (Application Programming Interface) is a defined set of rules that lets different software components communicate with each other.",
}


def general_response(query: str) -> str:
    """Give a direct answer for known topics, or a helpful generic reply otherwise
    (never claims the agent has 'no tool')."""
    query_lower = query.lower()
    for topic, answer in GENERAL_KNOWLEDGE_BASE.items():
        if topic in query_lower:
            return answer
   
    stripped = query.strip().rstrip("?").strip()
    return f"Here's a general response: '{stripped}' is a broad topic -- could you narrow it down, or ask me to calculate something or extract keywords instead?"


def agent(query: str):
    """Route a query to the right tool and return a structured JSON-style response."""
    if not isinstance(query, str) or not query.strip():
        logger.error("Empty or invalid query received")
        return {"type": "error", "result": "Query must be a non-empty string"}

    query_lower = query.lower()

    try:
        # Route 1
        if looks_like_math(query, query_lower):
            logger.info("Routing to Calculator Tool for query: %r", query)
            expression = extract_expression(query)
            result = calculator(expression)
            if result == "Error in calculation":
                return {"type": "error", "result": f"Could not evaluate expression: {expression}"}
            return {"type": "calculation", "result": result}

        # Route 2
        elif looks_like_keyword_request(query_lower):
            logger.info("Routing to Keyword Extractor Tool for query: %r", query)
            text = clean_text_for_keywords(query)
            keywords = extract_keywords(text)
            return {"type": "keywords", "result": keywords}

        # Route 3
        else:
            logger.info("Routing to General Response for query: %r", query)
            return {"type": "general", "result": general_response(query)}

    except Exception as e:
        logger.exception("Agent failed while processing query: %r", query)
        return {"type": "error", "result": f"Something went wrong: {e}"}

## Expected Output Format

```
{
  "type": "calculation / keywords / general / error",
  "result": ...
}
```

In [17]:
# Test Cases

queries = [
    "Calculate 90 + 5",
    "Extract keywords from Artificial Intelligence is transforming the world",
    "What is python",
    "What is 28+5?",                      
    "22 * 3 - 7",                           
    "Give me the key terms from this project report",  
    "",                                    
    "Calculate car",                      
]

for q in queries:
    print("Query:", repr(q))
    print("Response:", agent(q))
    print("-" * 50)

2026-08-09 09:32:28,636 | INFO | Routing to Calculator Tool for query: 'Calculate 90 + 5'
2026-08-09 09:32:28,643 | INFO | Routing to Keyword Extractor Tool for query: 'Extract keywords from Artificial Intelligence is transforming the world'
2026-08-09 09:32:28,647 | INFO | Routing to General Response for query: 'What is python'
2026-08-09 09:32:28,650 | INFO | Routing to Calculator Tool for query: 'What is 28+5?'
2026-08-09 09:32:28,653 | INFO | Routing to Calculator Tool for query: '22 * 3 - 7'
2026-08-09 09:32:28,657 | INFO | Routing to Keyword Extractor Tool for query: 'Give me the key terms from this project report'
2026-08-09 09:32:28,661 | ERROR | Empty or invalid query received
2026-08-09 09:32:28,664 | INFO | Routing to General Response for query: 'Calculate car'


Query: 'Calculate 90 + 5'
Response: {'type': 'calculation', 'result': '95'}
--------------------------------------------------
Query: 'Extract keywords from Artificial Intelligence is transforming the world'
Response: {'type': 'keywords', 'result': ['transforming', 'artificial', 'intelligence', 'world']}
--------------------------------------------------
Query: 'What is python'
Response: {'type': 'general', 'result': 'Python is a high-level, general-purpose programming language known for its readability and wide use in AI, web development, and automation.'}
--------------------------------------------------
Query: 'What is 28+5?'
Response: {'type': 'calculation', 'result': '33'}
--------------------------------------------------
Query: '22 * 3 - 7'
Response: {'type': 'calculation', 'result': '59'}
--------------------------------------------------
Query: 'Give me the key terms from this project report'
Response: {'type': 'keywords', 'result': ['project', 'report']}
--------------------

In [18]:
# Interactive Mode

while True:
    user_input = input("Enter query (type 'exit' to stop): ")
    if user_input.lower() == "exit":
        break
    print("Response:", agent(user_input))

2026-08-09 09:32:59,940 | INFO | Routing to General Response for query: 'what is python'


Response: {'type': 'general', 'result': 'Python is a high-level, general-purpose programming language known for its readability and wide use in AI, web development, and automation.'}


2026-08-09 09:33:04,081 | INFO | Routing to Calculator Tool for query: '20+40'


Response: {'type': 'calculation', 'result': '60'}


2026-08-09 09:33:16,337 | INFO | Routing to Calculator Tool for query: 'calculate 40-9/50'


Response: {'type': 'calculation', 'result': '39.82'}


2026-08-09 09:34:00,719 | INFO | Routing to Keyword Extractor Tool for query: 'extract keywords from india is one of the beautiful contry in the world'


Response: {'type': 'keywords', 'result': ['beautiful', 'india', 'world', 'contry']}
